In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F

# Load

In [0]:
dbutils.widgets.text("silver_table_name", "tb_motivos")
silver_table_name = dbutils.widgets.get("silver_table_name")

df_motivos = load_df(silver_table_name)

# Atributos

In [0]:
# A dimensão expõe apenas as colunas de negócio da silver
df_atributos = df_motivos.select(
    F.col("id_motivo"),
    F.col("nome_motivo"),
    F.col("categoria"),
    F.col("criticidade")
)

# Chave

In [0]:
# A chave substituta isola a dimensão de mudanças no id de negócio e é criada na primeira coluna
df_dim_motivo = add_sequential_key(df_atributos, "sk_motivo")

# Verificação

In [0]:
# A dimensão deve ter uma linha por motivo e a chave deve ser única
total = df_dim_motivo.count()
motivos = df_dim_motivo.select("id_motivo").distinct().count()
chaves = df_dim_motivo.select("sk_motivo").distinct().count()
print(f"Linhas: {total} | id_motivo únicos: {motivos} | sk_motivo únicas: {chaves}")

# Salvamento

In [0]:
dbutils.widgets.text("gold_table_name", "dim_motivo")
gold_table_name = dbutils.widgets.get("gold_table_name")

save_df(df_dim_motivo, gold_table_name)